In [ ]:
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.8/786.8 kB 7.3 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.99.8
    Uninstalling openai-1.99.8:
      Successfully uninstalled openai-1.99.8


# 1. Générer des requêtes synthétiques

In [ ]:
import pandas as pd
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
import json
from datasets import load_dataset
from google.colab import userdata
from openai import AzureOpenAI



client = AzureOpenAI(
    azure_endpoint=userdata.get('AZURE_ENDPOINT'),
    api_key=userdata.get('AZURE_OPENAI_KEY'),
    api_version="2023-12-01-preview",
)

In [ ]:
prompt_template = lambda job_description : f"""Read the following job description and create a concise job search query with at most 3 specialized skills or \
areas of expertise that are distinct to the role. Exclude generic data science or software engineering skills like AI, machine \
learning, and coding languages unless they are explicitly highlighted as unique or advanced. Keep the query short and human-like, \
suitable for typing into a search engine.

Here's the job description: {job_description}"""


def generate_query(job_description):
    """
        Fonction permettant de générer une requête synthétique pour saisir la description du poste.
    """

    # gérérer le prompt
    prompt = prompt_template(job_description)

    # faire le call api
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature = 0.7
    )

    # retourner la réponse
    return response.choices[0].message.content

A. Charger les données

In [ ]:
# Charger les données de HF
ds = load_dataset("datastax/linkedin_job_listings")

# convertir dans un dataframe pandas
df = ds['train'].to_pandas()

# garder uniquement les titres et descriptions
df = df[['title', 'description']]
df.shape

postings.csv:   0%|          | 0.00/517M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/123849 [00:00<?, ? examples/s]

(123849, 2)

In [ ]:
# Liste des éléments textuels recherchés
search_terms = ["Data Scientist", "Data Analyst", "Machine Learning Engineer",
                "Data Engineer", "AI Engineer", "Deep Learning"]

# Créer un modèle d'expression régulière pour correspondre à n'importe laquelle des chaînes
pattern = '|'.join(search_terms)

# Filtrer les lignes qui contiennent l'un des termes de recherche
df = df[df['title'].str.contains(pattern, case=False, na=False)]
df.shape

(1179, 2)

In [ ]:
# sauvegarder le fichier
df.to_csv('job_data.csv')

B. Générer des requêtes

In [ ]:
df = df.iloc[:1100]

job_description_list = df['description'].to_list()

Approche sans patch :

In [ ]:
# Approche sans patch :
from tqdm import tqdm

synthetic_query_list = []
for job_description in tqdm(job_description_list):
    synthetic_query_list.append(generate_query(job_description).replace('"',''))

# Rajouter les requêtes à df
df['query'] = synthetic_query_list

df.to_csv('data/job_data_w_query.csv')

100%|██████████| 1100/1100 [2:37:05<00:00,  8.57s/it]


OSError: Cannot save file into a non-existent directory: 'data'

In [ ]:
df.to_csv('job_data_w_query.csv')

approche par batch :

In [ ]:
job_description_list = df['description'].to_list()


# Créer des requêtes par batch
batch_requests = [
    {
        "custom_id": f"request-{i+1}",  # Identifiant personnalisé pour le suivi
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": "gpt-4o",
            "messages": [
                {"role": "user", "content": prompt_template(job_description)}
            ],
            "temperature": 0.7
        }
    }
    for i, job_description in enumerate(job_description_list)
]


# Convertir au format JSONL (JSON délimité par des sauts de ligne)
batch_jsonl = "\n".join(json.dumps(request) for request in batch_requests)

In [ ]:
# Enregistrer dans un fichier .jsonl
with open("batch_requests.jsonl", "w") as file:
    file.write(batch_jsonl)

In [ ]:


# upload file using the file_client
batch_input_file = client.files.create(
    file=open("batch_requests.jsonl", "rb"),
    purpose="fine-tune" # Changed purpose to 'fine-tune'
)

print(batch_input_file)

FileObject(id='file-a7a681ea09e34dafaabb15b637dffd73', bytes=5041930, created_at=1753879178, filename='batch_requests.jsonl', object='file', purpose='fine-tune', status='pending', expires_at=None, status_details=None, updated_at=1753879178)
